# Unit 3 Exercise: The Factory Method Pattern for a Car Manufacturer

Ian Stevens, Advanced Object-Oriented Design and Programming, Unit 3.

The brief asks for a system that builds Sedans, SUVs and Hatchbacks without the main program naming their classes. It specifies an abstract `Car` with `drive()`, three concrete cars, an abstract `CarFactory` with the factory method `create_car()`, three concrete factories, and a demonstration. The code is in `cars.py` and the tests in `test_cars.py`. This notebook walks through the design, then looks at what the pattern does and does not buy in Python.

In [1]:
import abc
import ast
import inspect
import platform
import textwrap
import threading

import cars
from cars import (FACTORIES, SUV, Car, CarFactory, Hatchback, HatchbackFactory, Sedan,
                  SedanFactory, SerialNumbers, SUVFactory, factory_for, road_test,
                  run_production)

print("Python", platform.python_version(), platform.python_implementation())

Python 3.13.18 CPython


## 1. The four roles

The Gang of Four definition, as Sarcar (2022, ch. 3) quotes it, is that the pattern 'defines an interface for creating an object, but lets subclasses decide which class to instantiate'. It has four roles: a product, concrete products, a creator that declares the factory method, and concrete creators that override it. Each role is a class in `cars.py`. Python's `abc` module makes `Car` and `CarFactory` abstract, so neither can be instantiated and a subclass that forgets to override the abstract method cannot be instantiated either.

In [2]:
for cls in (Car, Sedan, SUV, Hatchback, CarFactory, SedanFactory, SUVFactory, HatchbackFactory):
    abstract = sorted(cls.__abstractmethods__)
    print(f"{cls.__name__:<17} parent={cls.__mro__[1].__name__:<11} abstract={abstract or '-'}")

for cls in (Car, CarFactory):
    try:
        cls()
    except TypeError as error:
        print(f"\n{cls.__name__}() -> TypeError: {error}")

Car               parent=ABC         abstract=['drive']
Sedan             parent=Car         abstract=-
SUV               parent=Car         abstract=-
Hatchback         parent=Car         abstract=-
CarFactory        parent=ABC         abstract=['create_car']
SedanFactory      parent=CarFactory  abstract=-
SUVFactory        parent=CarFactory  abstract=-
HatchbackFactory  parent=CarFactory  abstract=-

Car() -> TypeError: Can't instantiate abstract class Car without an implementation for abstract method 'drive'

CarFactory() -> TypeError: Can't instantiate abstract class CarFactory without an implementation for abstract method 'create_car'


## 2. The demonstration

`main()` is the demonstration the brief asks for. It turns each name in an order into a factory with `factory_for()` and hands the factories to `run_production()`, which only knows about `CarFactory` and `Car`.

In [3]:
cars.main()
print()
cars.main(["hatchback", "hatchback", "SUV"])

CAR-000001 Sedan: a smooth, quiet ride on the motorway
CAR-000002 SUV: high driving position and four-wheel drive on a gravel track
CAR-000003 Hatchback: nipping through town and into a tight parking space

CAR-000001 Hatchback: nipping through town and into a tight parking space
CAR-000002 Hatchback: nipping through town and into a tight parking space
CAR-000003 SUV: high driving position and four-wheel drive on a gravel track


The client functions refer to no concrete class. Walking their syntax trees confirms it:

In [4]:
concrete = {"Sedan", "SUV", "Hatchback", "SedanFactory", "SUVFactory", "HatchbackFactory"}

def names_in(function):
    tree = ast.parse(textwrap.dedent(inspect.getsource(function)))
    return {node.id for node in ast.walk(tree) if isinstance(node, ast.Name)}

for function in (road_test, run_production, CarFactory.produce, cars.main):
    print(f"{function.__qualname__:<18} concrete classes named: {sorted(names_in(function) & concrete) or 'none'}")
print(f"{'FACTORIES':<18} maps {dict((k, v.__name__) for k, v in FACTORIES.items())}")

road_test          concrete classes named: none
run_production     concrete classes named: none
CarFactory.produce concrete classes named: none
main               concrete classes named: none
FACTORIES          maps {'sedan': 'SedanFactory', 'suv': 'SUVFactory', 'hatchback': 'HatchbackFactory'}


## 3. What the creator adds

With only `create_car()`, a `CarFactory` would be a constructor with extra steps: `SedanFactory().create_car()` does nothing that `Sedan()` does not. The pattern pays off when the creator has an operation of its own that uses the product. Sarcar (2022, ch. 3) makes the same move in his second demonstration, where `createAndDisplayAnimal()` calls the factory method and then runs code common to every product. Here that operation is `produce()`, written once against `Car`: it calls the factory method, refuses anything that is not a `Car`, and stamps a serial number from a `SerialNumbers` source that is passed in.

In [5]:
print(inspect.getsource(CarFactory.produce))

serials = SerialNumbers()
for factory in (SedanFactory(serials), SUVFactory(serials), SedanFactory(serials)):
    record = factory.produce()
    print(record.serial, record.factory, "->", record.car.describe())

    def produce(self) -> BuildRecord:
        """Make a car with the factory method, check it and give it a serial."""
        car = self.create_car()
        if not isinstance(car, Car):
            raise TypeError(
                f"{type(self).__name__}.create_car() returned "
                f"{type(car).__name__}, not a Car"
            )
        return BuildRecord(self._serials.next(), car, type(self).__name__)

CAR-000001 SedanFactory -> Sedan with 5 seats
CAR-000002 SUVFactory -> SUV with 7 seats
CAR-000003 SedanFactory -> Sedan with 5 seats


In [6]:
class BrokenFactory(CarFactory):
    def create_car(self):
        return "a car, honestly"

try:
    BrokenFactory().produce()
except TypeError as error:
    print("TypeError:", error)

TypeError: BrokenFactory.create_car() returned str, not a Car


The check is a run-time backstop. `mypy --strict` passes on `cars.py`, and it would flag `BrokenFactory` before it ran, because `create_car()` is annotated to return `Car`.

There is a gap the pattern cannot close in Python. Nothing stops a caller writing `Sedan()` directly and skipping `produce()`, its check and its serial number. Oliveira (2023) makes constructors private in C++ so that the factory is the only way in, and Soshin (2024) does the same in Kotlin. Python has no private constructors, so here the factory is a convention, the same limit Unit 2 found with template methods.

In [7]:
car = Sedan()
print(car.drive(), "| serial number:", getattr(car, "serial", "none"))

Sedan: a smooth, quiet ride on the motorway | serial number: none


## 4. Where the decision goes

The brief says the main program should create cars without specifying their classes. Something still has to choose `SedanFactory` over `SUVFactory`. In this design that choice sits in one read-only table, `FACTORIES`, used by `factory_for()`. That is a string-keyed simple factory of the kind Sarcar (2022, ch. 2) and Reddy (2024, ch. 3) describe, with Reddy's registry so that it can be extended. The Factory Method keeps the client free of concrete classes; it does not remove the decision, it moves it to the place where the program is assembled.

A new car type therefore needs a product class, a factory class and a registry entry. Passing a registry in means none of them needs `cars.py` to change.

In [8]:
class ElectricCar(Car):
    body_type, seats = "Electric", 4
    def drive(self):
        return "Electric: silent acceleration away from the lights"

class ElectricCarFactory(CarFactory):
    def create_car(self):
        return ElectricCar()

registry = dict(FACTORIES, electric=ElectricCarFactory)
serials = SerialNumbers()
print(run_production(factory_for(name, serials, registry) for name in ["electric", "sedan"]))

try:
    FACTORIES["electric"] = ElectricCarFactory
except TypeError as error:
    print("The default registry is read-only:", error)

['CAR-000001 Electric: silent acceleration away from the lights', 'CAR-000002 Sedan: a smooth, quiet ride on the motorway']
The default registry is read-only: 'mappingproxy' object does not support item assignment


## 5. A Python comparison: a class is already a factory

In Python a class is a callable that returns an instance, so the three factory classes can be replaced by the car classes themselves, or by a dictionary of them. The client code below is just as free of concrete names. What it loses is the creator's own operation: there is nowhere to put the check and the serial number except in every caller, or in a function that is `produce()` under another name.

In [9]:
from collections.abc import Callable

def road_test_callable(make_car: Callable[[], Car]) -> str:
    return make_car().drive()

for make_car in (Sedan, SUV, Hatchback):
    print(road_test_callable(make_car))

lines_of_code = {cls.__name__: len(inspect.getsource(cls).splitlines())
                 for cls in (CarFactory, SedanFactory, SUVFactory, HatchbackFactory)}
print("\nLines spent on the factory hierarchy:", lines_of_code, "total", sum(lines_of_code.values()))

Sedan: a smooth, quiet ride on the motorway
SUV: high driving position and four-wheel drive on a gravel track
Hatchback: nipping through town and into a tight parking space

Lines spent on the factory hierarchy: {'CarFactory': 23, 'SedanFactory': 3, 'SUVFactory': 3, 'HatchbackFactory': 3} total 32


Soshin (2024) makes the same argument for Kotlin: the `object` keyword covers Singleton, data class `copy()` covers Prototype, and default and named arguments remove most of the need for Builder. The pattern still names the idea, but the language does part of the work.

## 6. A change to the product still breaks the subclasses

The factories protect the client from concrete classes, not from changes to `Car` itself. Reddy (2024, ch. 10) warns that adding a pure virtual method to a published abstract base class breaks every client subclass. Python behaves the same way, but the break appears only when a subclass is instantiated, so tests are the only early warning.

In [10]:
class CarV2(Car):
    @abc.abstractmethod
    def refuel(self): ...

class OldSedan(CarV2):
    body_type, seats = "Sedan", 5
    def drive(self):
        return "Sedan: written against version 1"

print("Class definition accepted:", OldSedan.__name__)
try:
    OldSedan()
except TypeError as error:
    print("Instantiation fails:", error)

class CarV2WithDefault(Car):
    def refuel(self):
        return "refuelled at the pump"

class OldSedan2(CarV2WithDefault):
    body_type, seats = "Sedan", 5
    def drive(self):
        return "Sedan: written against version 1"

print("With a default implementation:", OldSedan2().refuel())

Class definition accepted: OldSedan
Instantiation fails: Can't instantiate abstract class OldSedan without an implementation for abstract method 'refuel'
With a default implementation: refuelled at the pump


## 7. Serial numbers across threads

The factories share one `SerialNumbers` source, which is shared mutable state. Reddy (2024, ch. 9) recommends taking a lock only in the public method (the Thread-Safe Interface pattern), which is what `SerialNumbers.next()` does. Nine factories each building 300 cars in parallel threads produce no repeated numbers.

In [11]:
serials = SerialNumbers()
factories = [maker(serials) for maker in FACTORIES.values()] * 3
made, guard = [], threading.Lock()

def build(factory):
    batch = [factory.produce().serial for _ in range(300)]
    with guard:
        made.extend(batch)

threads = [threading.Thread(target=build, args=(f,)) for f in factories]
for t in threads: t.start()
for t in threads: t.join()
print(f"{len(made)} cars built, {len(set(made))} unique serial numbers, last issued {max(made)}")

2700 cars built, 2700 unique serial numbers, last issued CAR-002700


## 8. Running the tests

In [12]:
import io
import unittest
import test_cars

suite = unittest.defaultTestLoader.loadTestsFromModule(test_cars)
result = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(suite)
print(f"Ran {result.testsRun} tests: {len(result.failures)} failures, {len(result.errors)} errors")
for case in test_cars.unittest.defaultTestLoader.getTestCaseNames(test_cars.FactoryContract):
    print(" ", case)

Ran 26 tests: 0 failures, 0 errors
  test_create_car_returns_a_car
  test_create_car_returns_a_new_object_each_time
  test_factories_with_their_own_source_do_not_share_numbers
  test_produce_returns_a_build_record
  test_shared_serial_source_gives_unique_increasing_serials


## References

Oliveira, C. (2023) *Options and derivatives programming in C++23: algorithms and programming techniques for the financial industry*. 3rd edn. Berkeley, CA: Apress.

Reddy, M. (2024) *API design for C++*. 2nd edn. San Diego: Elsevier Science and Technology.

Sarcar, V. (2022) *Java design patterns: a hands-on experience with real-world examples*. 3rd edn. Berkeley, CA: Apress.

Soshin, A. (2024) *Kotlin design patterns and best practices*. 3rd edn. Birmingham: Packt Publishing.